# Predikcia vývoja COVID-19 pomocou analýzy časových radov a strojového učenia

# 1. Úvod

Tento projekt sa zameriava na predikciu vývoja počtu prípadov ochorenia COVID-19. Využíva dáta z prvých mesiacov pandémie, kedy neexistovali vakcíny ani plošné obmedzenia. Sústredí sa na predpovedanie krátkodobých aj dlhodobých trendov. Tieto predpovede môžu slúžiť ako podklad pre tvorbu stratégií a na efektívne riadenie situácie v oblasti zdravotníctva.

Projekt slúži ako praktická ukážka mojich zručností v oblasti analýzy časových radov a prediktívnej analytiky. Vyvíjam v ňom modely, ktoré dokážu identifikovať a modelovať trendy a sezónnosť v dátach. Projekt je rozdelený do dvoch samostatných  nadväzujúcich notebookov Predikcia COVID 19 a LTSM covid 19. Pre komplexnú analýzu a model je potrebné najprv spustiť prvý notebook.


**Postup projektu je nasledovný:**

*Príprava dát:* Načítanie a čistenie historických dát.

*Modelovanie:* Implementácia a porovnanie dvoch typov modelov:

Štatistické modely (ARIMA/SARIMA): Na predikciu trendov a sezónnosti.

Modely strojového učenia (LSTM): Využitie hlbokých neurónových sietí na zachytenie komplexných vzorov v dátach.

Vizualizácia: Prezentácia výsledkov a porovnanie presnosti jednotlivých modelov.

Cieľom projektu je nielen vytvoriť presnú predpoveď, ale tiež demonštrovať, ako možno kombináciou rôznych prístupov získať robustný a spoľahlivý model.

# 2. Analýza a spracovanie dát (Data Preprocessing)
V tejto úvodnej fáze projektu sa zameriavam na prípravu a spracovanie surových dát. Tieto kroky sú kritické pre zabezpečenie spoľahlivosti a presnosti následných analýz a predikcií.

**Kód nižšie demonštruje nasledujúce procesy:**

*Načítanie a filtrovanie dát:* Dáta o potvrdených prípadoch COVID-19 sú načítané z verejne dostupného zdroja na GitHube. Následne sú dáta filtrované a vybrané sú iba tie, ktoré sa týkajú Slovenska, čo umožňuje cielenú analýzu.

*Príprava časového radu:* Stĺpec s dátumom je nastavený ako index. To je nevyhnutný krok pre analýzu časových radov a následné modelovanie.

*Čistenie dát:* Z dátovej sady sú odstránené nepotrebné stĺpce a všetky riadky, ktoré obsahujú chýbajúce hodnoty.

Výsledkom tejto fázy je čistý a usporiadaný súbor dát, pripravený na použitie v prediktívnych modeloch.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# URL adresa datasetu
url = 'https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv'

# Načítanie dát
try:
    df = pd.read_csv(url, parse_dates=['date'])
    print("Dáta boli úspešne načítané.")
except Exception as e:
    print(f"Nastala chyba pri načítaní dát: {e}")
    df = pd.DataFrame()

if not df.empty:
    # Vyber iba stĺpce, ktoré potrebujeme
    df = df[['date', 'location', 'new_cases']]

    # Vyber dát pre konkrétnu krajinu
    country = 'Slovakia'
    df_country = df[df['location'] == country].copy()

    # Nastavenie stĺpca s dátumom ako index
    df_country.set_index('date', inplace=True)

    # Odstránenie nepotrebnných stĺpccov
    df_country.drop('location', axis=1, inplace=True)

    # Odstránenie riadkov s chýbajúcimi hodnotami
    df_country.dropna(inplace=True)

    print("\n--- Prvých 5 riadkov upraveného datasetu ---")
    print(df_country.head())

    print(f"\n--- Počet riadkov pre {country} po úprave ---")
    print(len(df_country))

    # Vykreslenie časového radu
    df_country.plot(figsize=(15, 6))
    plt.title(f'Denný počet nových prípadov COVID-19 na {country}')
    plt.xlabel('Dátum')
    plt.ylabel('Počet prípadov')
    plt.show()

#3.  Test stacionarity dát (Augmented Dickey-Fuller Test)
Predtým, ako použijeme štatistické modely na predikciu časových radov, je dôležité, aby boli dáta stacionárne. To znamená, že štatistické vlastnosti dát (priemer, smerodajná odchýlka) sa v priebehu času nemenia. Ak dáta nie sú stacionárne, predikcie modelov môžu byť nespoľahlivé.

Na testovanie stacionarity som použila tzv. diferencovanie a Augmented Dickey-Fuller (ADF) test.

**Diferencovanie:** Tento proces spočíva v odčítaní predchádzajúcej hodnoty od aktuálnej. Odstránili sme tým hlavný trend v dátach, čím sme ich priblížili stacionarite.

**ADF test:** Ide o štatistický test, ktorý preukazuje, či sú dáta stacionárne. Ak je p-hodnota (p-value) menšia ako 0.05, môžeme s vysokou istotou predpokladať, že dáta sú stacionárne a sú pripravené na modelovanie.

Výsledky testu, ktoré sú uvedené nižšie, potvrdzujú, že naše upravené dáta sú stacionárne.



In [ ]:
from statsmodels.tsa.stattools import adfuller


# Uistime sa, že 'df_diff' je  diferencovaný DataFrame
# Vytvoríme ho pre istotu znova, aby sme eliminovali prípadnú chybu
df_country['diff_cases'] = df_country['new_cases'].diff(periods=1)
df_diff = df_country['diff_cases'].dropna()


print("--- Výsledky Augmented Dickey-Fuller Testu ---")
dftest = adfuller(df_diff, autolag='AIC')
dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
for key,value in dftest[4].items():
    dfoutput['Critical Value (%s)'%key] = value
print(dfoutput)

# 4.Rozdelenie dát na trénovaciu a testovaciu sadu
Pred samotným modelovaním je nevyhnutné rozdeliť dáta, aby sme vedeli vyhodnotiť presnosť našich modelov. Tento krok zabezpečuje, že modely budú testované na dátach, ktoré ešte nevideli, čím overíme ich schopnosť spoľahlivo predpovedať.

V tomto kroku sme diferencované dáta rozdelili do dvoch sád:

Trénovacia sada (80% dát): Použije sa na natrénovanie modelov ARIMA.

Testovacia sada (20% dát): Slúži na vyhodnotenie, ako presne modely predpovedali vývoj nových prípadov.

In [ ]:


# Predpokladáme, že 'df_diff' je už vytvorený a obsahuje diferencované dáta

# Rozdelenie dát
train_size = int(len(df_diff) * 0.8)
train_data = df_diff.iloc[:train_size]
test_data = df_diff.iloc[train_size:]

print("--- Rozdelenie diferencovaných dát ---")
print(f"Veľkosť trénovacej sady: {len(train_data)} dní")
print(f"Veľkosť testovacej sady: {len(test_data)} dní")
print("\nTrénovacia sada (posledných 5 riadkov):")
print(train_data.tail())
print("\nTestovacia sada (prvých 5 riadkov):")
print(test_data.head())

# 5. Identifikácia parametrov modelu (ACF a PACF)
Na nastavenie správnych parametrov pre modely ARIMA a SARIMA je nevyhnutné analyzovať autokorelačné funkcie dát. Tieto grafy nám pomáhajú určiť, aká silná je závislosť medzi aktuálnymi a minulými hodnotami časového radu.

ACF (Autocorrelation Function): Graf autokorelačnej funkcie meria koreláciu medzi pozorovaním a jeho minulými hodnotami (lagmi). Pomáha nám identifikovať, ako sa dáta navzájom ovplyvňujú v priebehu času, a odhadnúť hodnoty AR (Autoregressive) a MA (Moving Average) parametrov.

PACF (Partial Autocorrelation Function): Graf parciálnej autokorelačnej funkcie ukazuje koreláciu medzi pozorovaním a jeho minulosťou, avšak eliminuje vplyv medziriadkových hodnôt. Tento graf je kľúčový pre správne určenie parametrov AR.

Na základe analýzy týchto grafov sa dajú určiť optimálne parametre modelu a zabezpečiť tak presnejšia predikcia.

In [ ]:

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Predpokladáme, že 'df_diff' je už  diferencovaný DataFrame
# Vykreslenie ACF grafu
plt.figure(figsize=(15, 6))
plot_acf(df_diff, lags=30, alpha=0.05)
plt.title('Autokorelačná funkcia (ACF)')
plt.xlabel('Počet lagov')
plt.ylabel('Korelácia')
plt.show()

# Vykreslenie PACF grafu
plt.figure(figsize=(15, 6))
plot_pacf(df_diff, lags=30, alpha=0.05)
plt.title('Parciálna autokorelačná funkcia (PACF)')
plt.xlabel('Počet lagov')
plt.ylabel('Korelácia')
plt.show()

# 6. Trénovanie a predikcia s modelom ARIMA
Po príprave dát a identifikácii parametrov pre model nastal čas na samotné modelovanie. V tejto časti som sa zamerala na vytvorenie a trénovanie ARIMA modelu, ktorý je ideálny na predikciu hodnôt v časových radoch.

Kód nižšie demonštruje nasledujúce kroky:

Vytvorenie a trénovanie modelu: Model ARIMA bol vytvorený s parametrami (1,1,1), ktoré boli určené na základe predchádzajúcej analýzy (ACF a PACF grafov). Tento model bol následne natrénovaný na trénovacej sade dát.

Zhrnutie výsledkov (Summary): Prehľadné zhrnutie modelu poskytuje dôležité štatistické informácie o jeho parametroch a celkovej spoľahlivosti.

Predikcia na testovacej sade: Model bol použitý na predpoveď budúcich hodnôt a tieto predikcie boli vizualizované a porovnané s reálnymi hodnotami z testovacej sady.

Tento proces ukazuje, ako možno pomocou štatistických modelov spoľahlivo predpovedať budúci vývoj na základe historických dát.

In [ ]:

import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA

# Predpokladáme, že 'train_data' je už vytvorená

# Vytvorenie a trénovanie ARIMA modelu
# Parametre p, d, q sú (1, 1, 1)
model_arima = ARIMA(
    train_data,
    order=(1, 1, 1),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results_arima = model_arima.fit()

# Zobrazenie súhrnu modelu
print("--- Súhrn ARIMA modelu ---")
print(results_arima.summary())

# Predikcie na testovacej sade
forecast_arima = results_arima.get_forecast(steps=len(test_data))
predicted_values_arima = forecast_arima.predicted_mean

print("\n--- Prvých 5 predikovaných hodnôt ---")
print(predicted_values_arima.head())

In [ ]:

import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA

# Predpokladáme, že 'train_data' je už vytvorená

# Vytvorenie a trénovanie ARIMA modelu
# Parametre p, d, q sú (1, 1, 1)
model_arima = sm.tsa.statespace.SARIMAX(
    train_data,
    order=(1, 1, 1),
    seasonal_order=(0, 0, 0, 0), # Explicitne nastavíme sezónne parametre na nulu
    enforce_stationarity=False,
    enforce_invertibility=False
)

results_arima = model_arima.fit()

# Zobrazenie súhrnu modelu
print("--- Súhrn ARIMA modelu ---")
print(results_arima.summary())

# Teraz urobíme predikcie na testovacej sade
forecast_arima = results_arima.get_forecast(steps=len(test_data))
predicted_values_arima = forecast_arima.predicted_mean

print("\n--- Prvých 5 predikovaných hodnôt ---")
print(predicted_values_arima.head())

# 7. Vizualizácia a vyhodnotenie výkonu modelu ARIMA
Po natrénovaní a predikcii je kľúčové vizuálne aj numericky vyhodnotiť, aký presný bol náš model. Táto časť projektu slúži na porovnanie predikovaných hodnôt s reálnymi, historickými dátami.

**V nasledujúcom kóde sme vykonali tieto kroky:**

*Vizualizácia predikcií:* Na grafe je zobrazený rozdiel medzi skutočnými (zelená čiara) a predpovedanými (červená čiara) hodnotami. Vizualizácia je kľúčová, pretože nám umožňuje rýchlo posúdiť, ako dobre sa model prispôsobil trendu v dátach.

*Numerické vyhodnotenie:* Na kvantitatívne posúdenie presnosti modelu sme vypočítali dve bežné metriky:

*Mean Absolute Error (MAE):* Ukazuje priemernú absolútnu odchýlku medzi predpoveďou a skutočnou hodnotou.

*Root Mean Squared Error (RMSE):* Podobne ako MAE, aj táto metrika meria chybu modelu, ale silnejšie penalizuje väčšie odchýlky.

Výsledky ukazujú, aký presný je model v predpovedi budúcich hodnôt a poskytujú dôležitý základ pre porovnanie s inými modelmi, ako je napríklad LSTM.

In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predpokladáme, že 'train_data', 'test_data' a 'predicted_values_arima' už sú definované

# Vykreslenie výsledkov
plt.figure(figsize=(15, 6))
plt.plot(train_data.index, train_data, label='Trénovacia sada', color='blue')
plt.plot(test_data.index, test_data, label='Skutočné hodnoty (Test)', color='green')
plt.plot(predicted_values_arima.index, predicted_values_arima, label='ARIMA predikcie', color='red', linestyle='--')

plt.title('Porovnanie ARIMA predikcií s reálnymi hodnotami')
plt.xlabel('Dátum')
plt.ylabel('Diferencované prípady')
plt.legend()
plt.show()

# Na výpočet metrík ako MAE alebo RMSE potrebujeme predikcie
actual_values = test_data
mae = mean_absolute_error(actual_values, predicted_values_arima)
rmse = np.sqrt(mean_squared_error(actual_values, predicted_values_arima))

print(f"Mean Absolute Error (MAE) pre ARIMA: {mae:.2f}")
print(f"Root Mean Squared Error (RMSE) pre ARIMA: {rmse:.2f}")

In [ ]:

# Predpokladáme, že 'train_data' a 'test_data' už sú definované
# Vytvoríme nový SARIMA model s parametrami (1,1,1)x(1,1,1,7)
model_sarima = sm.tsa.statespace.SARIMAX(
    train_data,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results_sarima = model_sarima.fit()

# Zobrazenie súhrnu modelu
print("--- Súhrn SARIMA modelu ---")
print(results_sarima.summary())

# Generovanie predikcií
forecast_sarima = results_sarima.get_forecast(steps=len(test_data))
predicted_values_sarima = forecast_sarima.predicted_mean

# Vizuálna kontrola
plt.figure(figsize=(15, 6))
plt.plot(train_data.index, train_data, label='Trénovacia sada', color='blue')
plt.plot(test_data.index, test_data, label='Skutočné hodnoty (Test)', color='green')
plt.plot(predicted_values_sarima.index, predicted_values_sarima, label='SARIMA predikcie', color='purple', linestyle='--')

plt.title('Porovnanie SARIMA predikcií s reálnymi hodnotami')
plt.xlabel('Dátum')
plt.ylabel('Diferencované prípady')
plt.legend()
plt.show()

# Na výpočet metrík MAE a RMSE
actual_values = test_data
mae = mean_absolute_error(actual_values, predicted_values_sarima)
rmse = np.sqrt(mean_squared_error(actual_values, predicted_values_sarima))

print(f"Mean Absolute Error (MAE) pre SARIMA: {mae:.2f}")
print(f"Root Mean Squared Error (RMSE) pre SARIMA: {rmse:.2f}")